# 1 环境依赖

# 2 数据目录结构

假设在一个Experiment中所有的slides的图像存储在一个文件夹中，要求图像的后缀名保持一致，如.tif, .tiff, .ome.tif。对于每一张图像，进行独立的Tile和Channel分离，生成的FOV保存在这张图像同名的文件夹中。
```
Experiment Folder
|
|---Slides1.ome.tif
|---Slides2.ome.tif
|---Slides3.ome.tif
...
```
在该步骤中，Tile产生的FOV满足以下目录结构
```
Experiment Folder
|
|---Slides1.ome.tif
|---Slides2.ome.tif
|---Slides3.ome.tif
...
|---Slides1
|   |---Tiles
|       |---FOV0.tif
|       |---FOV1.tif
|       ...
|   |---image_data
|       |---FOV0
|       |   |---channel1.tif
|       |   |---channel2.tif
|       |   |---channel3.tif
|       |   ...
|       |---FOV1
|       |   |---channel1.tif
|       |   |---channel2.tif
|       |   |---channel3.tif
|       |   ...
|---Slides2
|   |---Tiles
|       |---FOV0.tif
|       |---FOV1.tif
|       ...
|   |---image_data
|       |---FOV0
|       |   |---channel1.tif
|       |   |---channel2.tif
|       |   |---channel3.tif
|       |   ...
|       |---FOV1
|       |   |---channel1.tif
|       |   |---channel2.tif
|       |   |---channel3.tif
|       |   ...
|       |---FOV2
|       |   |---channel1.tif
|       |   |---channel2.tif
|       |   |---channel3.tif
|       |   ...
```

# 3 处理逻辑
## Tile
因为Qupath中的Tile会产生无组织的区域和尺寸不足1024\*1024的区域，所以在Qupath中生成FOV以后还需要人工核查，手动删除无组织的FOV，并用代码补全尺寸不足1024\*1024的图像。但是在Python当中应该不存在这样的问题，所以只需要写一个简单的Tile函数就可以了，把用户输入的图像的全片进行Tile，同时记录每一张FOV的位置信息，也就是在原始WSI中的x,y坐标信息，然后筛选没有组织信号的图像进行舍弃。

> ### Parameter
> size: int, define the size of FOV in pixel level. size = 1024 or 512 or 128.
> 
> overlap: int, define the overlay between adjacent tile in pixel level, usually take ~10% of the size.
>
> ### Input
> ./Experiment Folder
>
> ### Output
> ./Experiment Folder/Slides Name/Tiles/FOVx.tif
> 
> ./Experiment Fodler/Slides Name/fov_coordinates.csv

## Split Channels
根据用户输入的图像读取图像携带的metadata信息，可以选择输出图像的metadata信息，随后由用户指定各channel的名称`Channel Name`。

> ### Input
> ./Experiment Folder/Slides Name/Tiles/FOVx.tif
>
> ### Output
> ./Experiment Folder/Slides Name/FOVx/channelx.tif


# 4 原始代码


In [ ]:
import os
import numpy as np
import pandas as pd
import tifffile
from pathlib import Path

def is_tissue(patch, threshold=10):
    """
    判断当前 FOV 是否包含有效组织信号。
    这里使用一个简单的均值/标准差过滤。在实际应用中，
    你可能需要根据特定的 DAPI 或自发荧光背景来调整这里的逻辑。
    """
    # 如果图像全黑或方差极小，则判定为无组织
    if np.std(patch) < threshold:
        return False
    return True

def tile_image(image_path, base_experiment_dir, size=1024, overlap=102, tissue_threshold=10):
    """
    对整图进行 Tile 切片，剔除无组织的区域，并记录坐标。
    """
    image_path = Path(image_path)
    slide_name = image_path.name.split('.')[0] # 提取 Slides1 等名称
    
    # 构建输出目录
    slide_dir = Path(base_experiment_dir) / slide_name
    tiles_dir = slide_dir / "Tiles"
    tiles_dir.mkdir(parents=True, exist_ok=True)
    
    # 读取图像 (通常多通道 TIFF 形状为 C, Y, X)
    print(f"Loading {image_path}...")
    img = tifffile.imread(image_path)
    
    # 兼容性处理：确保 shape 顺序为 (C, H, W)
    if len(img.shape) == 2:
        img = img[np.newaxis, :, :] # 补充单通道维度
    elif img.shape[-1] < img.shape[0]: 
        # 如果通道在最后 (H, W, C)，则转置为 (C, H, W)
        img = np.transpose(img, (2, 0, 1))
        
    C, H, W = img.shape
    print(f"Image shape: {img.shape}, Data type: {img.dtype}")
    
    step = size - overlap
    fov_count = 0
    coordinates = []
    
    # 滑窗切片
    for y in range(0, H, step):
        for x in range(0, W, step):
            # 防止边界溢出，获取当前 patch
            y_end = min(y + size, H)
            x_end = min(x + size, W)
            
            patch = img[:, y:y_end, x:x_end]
            
            # 补全边缘尺寸不足的图像 (Padding 使得所有 FOV 均为 size x size)
            if patch.shape[1] < size or patch.shape[2] < size:
                pad_y = size - patch.shape[1]
                pad_x = size - patch.shape[2]
                patch = np.pad(patch, ((0, 0), (0, pad_y), (0, pad_x)), mode='constant', constant_values=0)
                
            # 检查是否有组织
            if is_tissue(patch, threshold=tissue_threshold):
                fov_name = f"FOV{fov_count}"
                fov_path = tiles_dir / f"{fov_name}.tif"
                
                # 保存为多通道 TIFF
                tifffile.imwrite(fov_path, patch, photometric='minisblack')
                
                # 记录坐标
                coordinates.append({
                    "FOV_Name": fov_name,
                    "X": x,
                    "Y": y,
                    "Width": size,
                    "Height": size
                })
                fov_count += 1

    # 保存坐标到 CSV
    coords_df = pd.DataFrame(coordinates)
    csv_path = slide_dir / "fov_coordinates.csv"
    coords_df.to_csv(csv_path, index=False)
    print(f"Tile generation complete for {slide_name}. Valid FOVs: {fov_count}. Coordinates saved to {csv_path}")


def split_channels(slide_dir, channel_names):
    """
    读取已经切好的 Tile 图像，按通道进行分离并保存在独立的目录中。
    """
    slide_dir = Path(slide_dir)
    tiles_dir = slide_dir / "Tiles"
    image_data_dir = slide_dir / "image_data"
    
    if not tiles_dir.exists():
        print(f"Error: {tiles_dir} does not exist. Please run tile_image first.")
        return
        
    fov_files = list(tiles_dir.glob("*.tif"))
    print(f"Found {len(fov_files)} FOVs. Starting channel splitting...")
    
    for fov_file in fov_files:
        fov_name = fov_file.stem # 例如 "FOV0"
        fov_output_dir = image_data_dir / fov_name
        fov_output_dir.mkdir(parents=True, exist_ok=True)
        
        # 读取单个 FOV
        fov_img = tifffile.imread(fov_file)
        
        # 确保提供的通道名数量与实际通道数匹配
        if len(channel_names) != fov_img.shape[0]:
            print(f"Warning: Number of channel names ({len(channel_names)}) does not match image channels ({fov_img.shape[0]}) in {fov_name}")
            # 如果不匹配，就用默认的 channel1, channel2 等
            actual_channel_names = [f"channel{i+1}" for i in range(fov_img.shape[0])]
        else:
            actual_channel_names = channel_names
            
        # 分离并保存各个通道
        for c_idx in range(fov_img.shape[0]):
            channel_img = fov_img[c_idx, :, :]
            channel_file = fov_output_dir / f"{actual_channel_names[c_idx]}.tif"
            tifffile.imwrite(channel_file, channel_img, photometric='minisblack')
            
    print(f"Channel splitting complete for {slide_dir.name}.")


# ==========================================
# 5 运行 Pipeline 的示例代码
# ==========================================
def process_experiment(experiment_folder, channel_names, size=1024, overlap=102):
    experiment_dir = Path(experiment_folder)
    
    # 查找所有的 .tif 或 .ome.tif 文件 (这里假设在根目录下)
    slide_files = list(experiment_dir.glob("*.tif")) + list(experiment_dir.glob("*.ome.tif"))
    # 去重
    slide_files = list(set(slide_files))
    
    for slide_path in slide_files:
        print(f"\n--- Processing {slide_path.name} ---")
        # 1. 图像切片与坐标记录
        tile_image(
            image_path=slide_path, 
            base_experiment_dir=experiment_folder, 
            size=size, 
            overlap=overlap
        )
        
        # 2. 通道分离
        slide_name = slide_path.name.split('.')[0]
        slide_dir = experiment_dir / slide_name
        split_channels(
            slide_dir=slide_dir, 
            channel_names=channel_names
        )

# 如果直接运行此脚本
if __name__ == "__main__":
    # 配置参数
    EXPERIMENT_DIR = "./Experiment_Folder"
    # 根据你的面板 (Panel) 设定通道名称
    CHANNEL_NAMES = ["DAPI", "CD3", "CD4", "CD8", "PanCK"] 
    
    # 你可以取消注释下面的代码来实际执行
    # process_experiment(EXPERIMENT_DIR, CHANNEL_NAMES, size=1024, overlap=100)